# Bab 4: Memahami Data

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab04/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab04")
print("Folder kerja:", os.getcwd())

## Listing 4.1: Ringkasan pertama data anggota

`kode/bab04/ringkasan.py`

In [ ]:
import pandas as pd

DATA = "kode/data/"
pel = pd.read_csv(DATA + "pelanggan.csv", parse_dates=["tanggal_daftar"])
pel = pel.drop(columns=["segmen_laten", "belanja_bulan_depan"])
print(pel.shape)
print(pel.dtypes.value_counts().to_string())

num = ["umur", "frekuensi_90h", "recency_hari", "nilai_90h"]
ring = pel[num].describe().T
ring = ring[["mean", "std", "min", "50%", "max"]]
ring.loc["nilai_90h"] /= 1000            # dalam ribu rupiah
print(ring.round(1).to_string())

print(pel["tingkat"].value_counts().to_string())
print(pel["kota"].value_counts(normalize=True)
      .mul(100).round(1).head(4).to_string())
print(pel["kota"].mode()[0], pel["nilai_90h"].mode()[0])

## Listing 4.2: Bentuk sebaran atribut anggota

`kode/bab04/sebaran.py`

In [ ]:
import numpy as np
import pandas as pd

DATA = "kode/data/"
pel = pd.read_csv(DATA + "pelanggan.csv")
pel["nilai_90h"] = pel["nilai_90h"] / 1000
kol = ["umur", "frekuensi_90h", "recency_hari", "nilai_90h"]

bentuk = pd.DataFrame({  # nilai_90h dalam ribu rupiah
    "rata": pel[kol].mean(),
    "median": pel[kol].median(),
    "kemiringan": pel[kol].skew(),
    "keruncingan": pel[kol].kurt(),
})
print(bentuk.round(2).to_string())

x = pel["nilai_90h"]
q1, q3 = x.quantile([.25, .75])
iqr = q3 - q1
batas = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
print("Q1, Q3, IQR :", q1, q3, iqr)
print("di luar pagar:", int(((x < batas[0]) | (x > batas[1])).sum()))

positif = x[x > 0]
print("nilai nol:", int((x == 0).sum()))
print("kemiringan log10:", round(np.log10(positif).skew(), 2))

## Listing 4.3: Kuartet Anscombe

`kode/bab04/anscombe.py`

In [ ]:
import numpy as np

x = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5])
y = {
    "I":   [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68],
    "II":  [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
    "III": [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73],
    "IV":  [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
}
x4 = np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8])
for k, yy in y.items():
    xx = x4 if k == "IV" else x
    yy = np.array(yy)
    r = np.corrcoef(xx, yy)[0, 1]
    b1, b0 = np.polyfit(xx, yy, 1)
    print(f"{k:>3}: rata y={yy.mean():.2f} var y={yy.var(ddof=1):.2f} "
          f"r={r:.3f} y={b0:.2f}+{b1:.3f}x")

## Listing 4.4: Korelasi dan asosiasi pada data anggota

`kode/bab04/korelasi.py`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

DATA = "kode/data/"
pel = pd.read_csv(DATA + "pelanggan.csv")
kol = ["frekuensi_90h", "recency_hari", "nilai_90h", "porsi_promo",
       "keluhan_90h"]
singkat = dict(zip(kol, ["frek", "recency", "nilai", "promo", "keluhan"]))
print(pel[kol].rename(columns=singkat).corr().round(2).to_string())
print(pel[kol].corr(method="spearman")
      .loc["frekuensi_90h", "recency_hari"].round(2))

print(pel.groupby("churn")[["recency_hari", "porsi_promo"]]
      .mean().round(2).to_string())

tab = pd.crosstab(pel["tingkat"], pel["churn"])
print(tab.to_string())
chi2, p, dof, _ = stats.chi2_contingency(tab)
n = tab.values.sum()
v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))
print(f"chi2={chi2:.1f}  dof={dof}  p={p:.1e}  V={v:.2f}")

## Listing 4.5: Jarak dan kemiripan antaranggota

`kode/bab04/jarak.py`

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import distance

DATA = "kode/data/"
pel = pd.read_csv(DATA + "pelanggan.csv", index_col="id_anggota")
kol = ["frekuensi_90h", "recency_hari", "nilai_90h"]
tiga = pel.loc[["A00001", "A00002", "A00003"], kol]
print(tiga.to_string())

a, b = tiga.values[0], tiga.values[1]
print("Euclides :", round(distance.euclidean(a, b), 1))
print("Manhattan:", round(distance.cityblock(a, b), 1))
print("Chebyshev:", round(distance.chebyshev(a, b), 1))

z = (pel[kol] - pel[kol].mean()) / pel[kol].std()
za, zb = z.loc["A00001"].values, z.loc["A00002"].values
print("Euclides baku:", round(distance.euclidean(za, zb), 2))
print("kosinus:", round(1 - distance.cosine(za, zb), 2))

menu_a = {"K01", "M01", "H03", "N02"}
menu_b = {"K01", "M01", "K04"}
j = len(menu_a & menu_b) / len(menu_a | menu_b)
print("Jaccard:", round(j, 2))

## Listing 4.6: Pemeriksaan mutu data mentah

`kode/bab04/mutu.py`

In [ ]:
import pandas as pd

DATA = "kode/data/"
mentah = pd.read_csv(DATA + "pelanggan_mentah.csv")
print(mentah.shape)
kosong = mentah.isna().mean().mul(100).round(1)
print(kosong[kosong > 0].to_string())
print("duplikat penuh:", mentah.duplicated().sum())
print("duplikat id   :", mentah["id_anggota"].duplicated().sum())
import textwrap
print(textwrap.fill(", ".join(sorted(mentah["kota"].unique())), 60))
umur = pd.to_numeric(mentah["umur"], errors="coerce")
print("umur < 15 atau > 90:", int(((umur < 15) | (umur > 90)).sum()))
teks = pd.to_numeric(mentah["nilai_90h"], errors="coerce").isna()
print("nilai_90h bukan angka:", int(teks.sum()))
print(mentah.loc[teks, "nilai_90h"].head(2).tolist())